# GX23 — Autovalores: Brent vs aproximação de Beck (1992)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/gx23_autovalores.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição

O problema GX23 corresponde à placa plana com fluxo prescrito em $x=0$ (tipo 2) e convecção em $x=L$ (tipo 3). Os autovalores $\beta_m$ satisfazem a equação transcendental (Eq. 4.28):

$$\xi_m \tan(\xi_m) = H_2, \qquad \xi_m = \beta_m L, \quad H_2 = \frac{h_2 L}{k}$$

Este notebook compara os autovalores obtidos pelo **método de Brent** com os valores **aproximados** pelas fórmulas de Beck (1992), com e sem a correção iterativa de Newton-Raphson (Eq. 4.33).

## Bibliotecas

In [ ]:
import numpy as np
from scipy.optimize import brentq
import pandas as pd

## Parâmetro $H_2$

Altere o valor de `H2` para explorar diferentes regimes de convecção. As fórmulas de Beck cobrem:

| Fórmula | Validade | Equação no livro |
|---|---|---|
| Eq. (4.29) | $m=1$, $0 < H_2 \leq 2$ | primeiro modo, convecção fraca |
| Eq. (4.30) | $m \geq 2$, $H_2$ pequeno | modos superiores: $m=2$ ($H_2 \leq 5$), $m=3$ ($H_2 \leq 8$), etc. |
| Eq. (4.31) | $m \geq 1$, $H_2$ grande | todos os modos para convecção intensa |

In [ ]:
H2 = 2.0   # H2 = h2·L/k  — altere aqui para explorar diferentes casos
M  = 5     # número de autovalores

print(f'H2 = {H2:.4f}')

## Autovalores por Brent

A equação $\xi\tan(\xi) = H_2$ pode ser escrita como $f(\xi) = \xi\tan(\xi) - H_2 = 0$.

A tangente tem singularidades em $\xi = (m - \tfrac{1}{2})\pi$ e é positiva nos intervalos $\bigl((m-1)\pi,\; (m - \tfrac{1}{2})\pi\bigr)$. Para $H_2 > 0$, cada raiz $\xi_m$ está em um desses intervalos:

| $m$ | Intervalo de busca |
|---|---|
| 1 | $(\varepsilon,\; \pi/2 - \varepsilon)$ |
| 2 | $(\pi + \varepsilon,\; 3\pi/2 - \varepsilon)$ |
| 3 | $(2\pi + \varepsilon,\; 5\pi/2 - \varepsilon)$ |

In [ ]:
eps = 1e-10

def f_gx23(xi, H2):
    """Equação transcendental GX23: ξ·tan(ξ) - H2 = 0"""
    return xi * np.tan(xi) - H2

xi_brent = np.array([
    brentq(f_gx23, (m-1)*np.pi + eps, (m-1)*np.pi + np.pi/2 - eps, args=(H2,))
    for m in range(1, M+1)
])

print('Autovalores por Brent ξ_m = β_m · L:')
for m, xi in enumerate(xi_brent, 1):
    print(f'  ξ_{m} = {xi:.8f}')

## Aproximações de Beck (1992)

### Eq. (4.29) — $m=1$, $0 < H_2 \leq 2$

$$\xi_1 = \left[\frac{3H_2}{3+H_2}\left(1 - \frac{1}{45}\left(\frac{3H_2}{3+H_2}\right)^{\!2}\right)\right]^{1/2}$$

### Eq. (4.30) — $m \geq 2$, pequenos valores de $H_2$

$$\xi_m = \frac{(m-1)\pi}{2(H_2+3)}\left[2H_2 + 3 + 3\left(1 + \frac{4H_2(H_2+3)}{3(m-1)^2\pi^2}\right)^{\!1/2}\right]$$

válida para: $m=2$ ($H_2 \leq 5$); $m=3$ ($H_2 \leq 8$); $m=4$ ($H_2 \leq 11$); $m=5$ ($H_2 \leq 13$).

### Eq. (4.31) — $m \geq 1$, grandes valores de $H_2$

$$\xi_m = \frac{(2m-1)\pi H_2}{2(H_2+1)}\left[1 + \frac{[(2m-1)\pi]^2}{12(H_2+1)^3 + [(2m-1)\pi]^2(2H_2-1)}\right]$$

### Eq. (4.33) — Correção de Newton-Raphson (aplicada iterativamente)

Partindo do valor aproximado $\xi'_m$, aplica-se repetidamente:

$$\xi_m \leftarrow \xi_m - \frac{\xi_m\tan\xi_m - H_2}{\tan\xi_m + \xi_m\sec^2\!\xi_m}$$

O denominador é $F'(\xi) = \tan\xi + \xi\sec^2\xi$ (derivada de $F = \xi\tan\xi - H_2$).

In [ ]:
lim_j15 = {2: 5, 3: 8, 4: 11, 5: 13}   # validade da Eq. (4.30) por modo

def beck_xi23(m, H2):
    """Aproximação inicial de Beck para ξ_m de GX23."""
    if m == 1 and 0 < H2 <= 2:                          # Eq. (4.29)
        u = 3*H2 / (3 + H2)
        return np.sqrt(u * (1 - u**2 / 45))
    elif m >= 2 and H2 <= lim_j15.get(m, 5):            # Eq. (4.30)
        c = (m-1)*np.pi / (2*(H2+3))
        return c * (2*H2 + 3 + 3*(1 + 4*H2*(H2+3) / (3*(m-1)**2*np.pi**2))**0.5)
    else:                                                # Eq. (4.31)
        n = 2*m - 1
        num = (n*np.pi)**2
        den = 12*(H2+1)**3 + num*(2*H2-1)
        return n*np.pi*H2 / (2*(H2+1)) * (1 + num/den)

def newton_iter(xi0, H2, tol=1e-12, nmax=100):
    """Newton-Raphson iterativo — Eq. (4.33)."""
    xi = xi0
    for _ in range(nmax):
        F  = xi * np.tan(xi) - H2
        Fp = np.tan(xi) + xi / np.cos(xi)**2   # tan(ξ) + ξ·sec²(ξ)
        dxi = F / Fp
        xi -= dxi
        if abs(dxi) < tol:
            return xi
    return xi

xi_beck    = np.array([beck_xi23(m, H2)                  for m in range(1, M+1)])
xi_beck_nr = np.array([newton_iter(beck_xi23(m, H2), H2) for m in range(1, M+1)])

## Tabela de comparação

In [ ]:
erro_beck    = 100 * np.abs(xi_beck    - xi_brent) / xi_brent
erro_beck_nr = 100 * np.abs(xi_beck_nr - xi_brent) / xi_brent

df = pd.DataFrame({
    'm'             : range(1, M+1),
    'ξ_Brent'       : xi_brent,
    'ξ_Beck'        : xi_beck,
    'Erro Beck (%)' : erro_beck,
    'ξ_Beck+NR'     : xi_beck_nr,
    'Erro+NR (%)'   : erro_beck_nr,
})
df = df.set_index('m')

pd.set_option('display.float_format', '{:.6f}'.format)
print(f'H2 = {H2:.4f}\n')
print(df.to_string())

## Observações

- As fórmulas de Beck para GX23 são **muito precisas** em toda a faixa de validade: erros iniciais tipicamente abaixo de 0,1%.
- Após a correção de Newton-Raphson (Eq. 4.33), os valores convergem à **precisão de máquina** em poucas iterações para qualquer $H_2 > 0$.
- O denominador correto do NR é $\tan\xi + \xi\sec^2\xi$ (sinal positivo).